In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_experiment(model_name):

    print(f"\nLoading {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32
    ).to(device)

    model.train()

    # Freeze transformer layers
    for param in model.parameters():
        param.requires_grad = False

    for param in model.lm_head.parameters():
        param.requires_grad = True

    # Value head
    class ValueHead(nn.Module):
        def __init__(self, hidden_size):
            super().__init__()
            self.value = nn.Linear(hidden_size, 1)

        def forward(self, hidden_states):
            return self.value(hidden_states).mean()

    critic = ValueHead(model.config.hidden_size).to(device)

    optimizer = torch.optim.Adam(
        list(model.lm_head.parameters()) + list(critic.parameters()),
        lr=1e-5
    )

    clip_eps = 0.2

    def reward_function(text):
        r = 0
        if "learning" in text.lower():
            r += 1
        if len(text.split()) > 30:
            r += 1
        return r - 1

    prompts = [
        "Explain reinforcement learning.",
        "What is GPU optimization?",
        "Describe comparative learning."
    ]

    total_reward = 0

    for epoch in range(2):
        for prompt in prompts:

            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            generated = model.generate(**inputs, max_new_tokens=40)

            outputs = model(generated, output_hidden_states=True)
            logits = outputs.logits
            hidden = outputs.hidden_states[-1]

            log_probs = F.log_softmax(logits, dim=-1)
            selected = log_probs.gather(2, generated.unsqueeze(-1)).squeeze(-1)
            log_prob = selected.mean()

            value = critic(hidden)

            text = tokenizer.decode(generated[0], skip_special_tokens=True)
            reward = torch.tensor(reward_function(text), dtype=torch.float32).to(device)

            advantage = reward - value.detach()

            ratio = torch.exp(log_prob - log_prob.detach())
            clipped = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantage
            policy_loss = -torch.min(ratio*advantage, clipped)

            value_loss = 0.5 * (reward - value).pow(2)

            loss = policy_loss + value_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_reward += reward.item()

    avg_reward = total_reward / (len(prompts)*2)

    print(f"Average Reward for {model_name}: {avg_reward}")

    del model
    torch.cuda.empty_cache()

    return avg_reward

In [2]:
qwen_reward = run_experiment("Qwen/Qwen1.5-0.5B")


Loading Qwen/Qwen1.5-0.5B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Average Reward for Qwen/Qwen1.5-0.5B: 0.6666666666666666


In [4]:
print("\nFinal Comparison:")
print("Qwen Avg Reward :", qwen_reward)



Final Comparison:
Qwen Avg Reward : 0.6666666666666666


In [5]:
llama_reward = run_experiment("TinyLlama/TinyLlama-1.1B-Chat-v1.0")


Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Average Reward for TinyLlama/TinyLlama-1.1B-Chat-v1.0: -0.3333333333333333


In [6]:
print("\nFinal Comparison:")

print("LLaMA Avg Reward:", llama_reward)


Final Comparison:
LLaMA Avg Reward: -0.3333333333333333
